In [ ]:
from langchain_core.tools import tool
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息参数:city: 城市名称，如"北京"、"上海"返回:天气信息字符串"""
    return city + '晴天，温度15°C'

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)
# 绑定工具model_with
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke('徐州天气如何？')

# 检查 AI 是否要调用工具
if response.tool_calls:
    print('AI 想调用工具：', response.tool_calls)
else:
    print('AI 直接回答：',response.content)

In [ ]:
from langchain_core.messages import HumanMessage,ToolMessage
from langchain_core.tools import tool
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息参数:city: 城市名称，如"北京"、"上海"返回:天气信息字符串"""
    return city + '晴天，温度15°C'

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)
# 绑定工具model_with
model_with_tools = model.bind_tools([get_weather])
messages = [
    HumanMessage(content='今天北京天气如何')
]
response = model_with_tools.invoke(messages)
# 模型生成调用工具请求
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather':
        # 拼接出ToolMessage实例
        tool_response = ToolMessage(
            content= get_weather.invoke(input=tool_call['args']),
            tool_call_id= tool_call['id'],
            name= tool_call['name']
            )
        print(tool_response)

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city: str, units: str = 'celsius', include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报
    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp= 22 if units == 'celsius' else 72
    result = f'{city}当天气温: {temp} {'摄氏度' if units == 'celsius' else '华氏度'}'
    if include_forecast:
        result += '\n未来五天都是晴天'
    return result

rprint(convert_to_openai_tool(get_weather))

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city: str, units: str = 'celsius', include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp= 22 if units == 'celsius' else 72
    result = f'{city}当天气温: {temp} {'摄氏度' if units == 'celsius' else '华氏度'}'
    if include_forecast:
        result += '\n未来五天都是晴天'
    return result

rprint(convert_to_openai_tool(get_weather))



In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

@tool(name_or_callable='getweather_') # 方法名称自定义
def get_weather(city: str) -> str:
    """
    天气查询工具
    """
    return  f'{city}天气晴朗'
print(convert_to_openai_tool(get_weather))

In [ ]:
from pydantic import BaseModel

class weatherInput(BaseModel):
    city: str

print(weatherInput(city='北京'))

In [ ]:
from pydantic import BaseModel,Field
class weatherInput(BaseModel):
    city: str = Field(
        default='北京'
    )
print(weatherInput())

In [ ]:
from pydantic import BaseModel,Field
class weatherInput(BaseModel):
    city: str = Field(
        default= '北京',
        description= '城市'
    )
    include_forecast: bool = Field(
        default= False,
        description= '是否包含未来五日天气预报'
    )
print(weatherInput())

In [ ]:
from pydantic import BaseModel,Field
from langchain_core.tools import tool
from  langchain_core.utils.function_calling import convert_to_openai_tool
from typing import Literal

class weatherInput(BaseModel):
    city: str = Field(
        default= '北京',
        description= '城市'
    )
    unit: Literal['celsius','fahrenheit'] = Field(
        default= 'celsius',
        description= '气温单位'
    )
    include_forecast: bool = Field(
        default= False,
        description= '是否包含未来五日天气预报'
    )

@tool(args_schema= weatherInput)
def get_weather(city: str, unit: str ='celsius', include_forecast: bool= False) ->str:
    """获取当日天气，可选未来五日天气预报"""
    temp= 22 if unit == 'celsius' else 72
    result = f'{city}当天气温: {temp} {'摄氏度' if unit == 'celsius' else '华氏度'}'
    if include_forecast:
        result += '\n未来五天都是晴天'
    return result
convert_to_openai_tool(get_weather)

In [ ]:
from langchain_core.tools import tool
from  langchain_core.utils.function_calling import convert_to_openai_tool
from typing import Literal

weather_schema = {
    'type': 'object',
    'properties': {
        'location': {'type': 'string'},
        'city': {'type': 'string'},
        'units': {'type': 'string'},
        'include_forecast': {'type': 'boolean'},
    },
    'required': ['location', 'units', 'include_forecast'],
}

@tool(args_schema= weather_schema)
def get_weather(city: str, unit: str ='celsius', include_forecast: bool= False) ->str:
    """获取当日天气，可选未来五日天气预报"""
    temp= 22 if unit == 'celsius' else 72
    result = f'{city}当天气温: {temp} {'摄氏度' if unit == 'celsius' else '华氏度'}'
    if include_forecast:
        result += '\n未来五天都是晴天'
    return result

convert_to_openai_tool(get_weather)


In [ ]:

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage

# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)

class WeatherSchema(BaseModel):
    city: str = Field(
        default= '北京',
        description= '城市名称'
    )
    if_forecast: bool = Field(
        default= False,
        description= '是否包含明日天气预报'
    )

@tool('get_weather_and_forecast',description= '查询当日天气，可以包含明日天气预报',args_schema= WeatherSchema)
def get_weather(city: str, if_forecast: bool= False) -> str:
    result = f'{city} 今天天气不错'
    if if_forecast:
         result += '\n明天也不错'
    return  result


model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(content='今天杭州天气如何？明天呢？')]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather_and_forecast':
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)

# 相当于获取天气接口
@tool('get_weather_and_forecast',parse_docstring= True)
def get_weather(city: str = '北京', if_forecast: bool= False) -> str:
    """
     查询当日天气，可以包含明日天气预报

     Args:
         city: 城市名称
         if_forecast: 是否包含明日天气预报
    """
    result = f'{city} 今天天气不错'
    if if_forecast:
         result += '\n明天要下雨'
    return  result

# 工具调取获取天气接口
model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(content='今天杭州天气如何？明天呢？')]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather_and_forecast':
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model


# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)

# 定义股票查询工具
@tool(parse_docstring= True)
def get_stock_price(company: str,timeframe: str = 'today'):
    """
     获取指定公司的股票价格信息

     Args:
         company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
         timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        '苹果公司': {'today': 185.20,'week': 183.50, 'month': 180.75},
        '微软公司': {'today': 415.86,'week': 412.30, 'month': 405.42},
        '谷歌公司': {'today': 15.42,'week': 15.20, 'month': 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, '未知时间范围')
        return f'{company} {timeframe}价格: {price}美元'
    else:
        return f'未找到股票代码 {company} 的数据'

# 定义新闻搜索工具
@tool(parse_docstring= True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        '苹果公司': [
            '苹果发布新款iPhone，股价上涨3%',
            '苹果与欧盟达成反垄断和解协议',
            '苹果将在印度扩大生产规模'
        ],
        '微软公司': [
            '微软Azure云业务季度增长超预期',
            '微软完成对Nuance的收购',
            '微软推出新一代AI助手Copilot'
        ],
        '谷歌公司': [
            '谷歌发布新AI模型，性能提升20%',
            '谷歌与OpenAI合作，开发新的AI助手',
            '谷歌在欧洲展开AI研究项目'
        ]
    }
    new_list = mock_news.get(company, [f'未找到{company}的相关新闻'])
    return '\n'.join(new_list)


# 工具列表
tools = [get_stock_price,search_news]

# 绑定工具
model_with_tools = model.bind_tools(tools)
message_list = []
# 用户消息
human_message = HumanMessage(content='苹果公司今天的股价是多少？最近有什么新闻？')
message_list.append(human_message)

# 工具调用
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)

    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print('没有工具调用，直接返回答案')
        break

    # 如果有调用工具，处理工具调用响应
    # 开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call['name'] == 'get_stock_price':
            stock_result = get_stock_price.invoke(tool_call)
            print('stock_result', stock_result)
            message_list.append(stock_result)
        if tool_call['name'] == 'search_news':
            news_result = search_news.invoke(tool_call)
            print('news_result', news_result)
            message_list.append(news_result)

for msg in message_list:
    msg.pretty_print()

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model


# 1. 读取 .env 配置文件中的信息，相关的环境变量以 .env文件中的优先
load_dotenv(override=True)
AGNES_AI_API_KEY = os.getenv('AGNES_AI_API_KEY')
AGNES_AI_BASE_URL = os.getenv('AGNES_AI_BASE_URL')
AGNES_AI_MODEL = os.getenv('AGNES_AI_MODEL')

model = init_chat_model(
    model_provider='openai',
    model=AGNES_AI_MODEL,
    api_key=AGNES_AI_API_KEY,
    base_url=AGNES_AI_BASE_URL,
)

@tool(parse_docstring= True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'

@tool(parse_docstring= True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return '近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。'

model_with_tools = model.bind_tools([get_weather,get_news])
messages = [
    HumanMessage(content= '今天杭州天气如何？今天新闻是什么？别瞎编')
]

response = model_with_tools.invoke(messages)

messages.append(response)

for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_weather':
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
    elif tool_call['name'] == 'get_news':
        tool_msg = get_news.invoke(tool_call)
        messages.append(tool_msg)
    else:
        raise  Exception('不存在的工具')
final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()